# Train a mechanism-family detector on the public NullRabbit bundle corpus

A self-contained walkthrough of **using Bundle v1 data for training** with the public
[`NullRabbit/nr-bundles-public`](https://huggingface.co/datasets/NullRabbit/nr-bundles-public)
dataset:

> download bundles → read manifest + Parquet → featurise → train over the family taxonomy → **evaluate honestly (in-distribution *and* cross-chain)**

The interesting result is not the model (a stock gradient-boosted tree). It is that the shared,
chain-agnostic format + taxonomy make a cross-chain question **testable**: hold out an entire chain and
measure whether a detector trained on the others recovers *that chain's* attack families zero-shot. That
leave-one-chain-out number is the one that decides where a detector's autonomy is **earned**. See
[`docs/integration-bundles-taxonomy-autonomy.md`](https://github.com/NullRabbitLabs/nr-bundle-spec/blob/main/docs/integration-bundles-taxonomy-autonomy.md).

In [ ]:
!pip -q install huggingface_hub pandas pyarrow numpy scikit-learn

In [ ]:
import json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from huggingface_hub import snapshot_download

## 1. Download the bundles

Each bundle is a directory: `manifest.json` + up to five Parquet modalities keyed on a monotonic `t_ns`.
We pull only the manifests and Parquet (no pcap).

In [ ]:
root = Path(snapshot_download("NullRabbit/nr-bundles-public", repo_type="dataset",
                               allow_patterns=["*/*.parquet", "*/manifest.json"]))
print("bundles on disk:", len(list(root.glob("crp_*/manifest.json"))))

## 2. Featurise one bundle

Reduce the two load-bearing modalities — `host` telemetry and per-request `responses` — to a handful of
aggregates. NaNs are fine: the model is NaN-native, so a bundle that didn't capture a modality contributes
NaN, not a fake zero. `resp.amp` (response ÷ request bytes) is the defining signal of the `response_amp`
family; `rss_bytes.slope` catches `memory_amp`; `num_connections` catches `connection_exhaustion`.

In [ ]:
def _agg(df, col):
    if col not in df or df[col].dropna().empty:
        return {}
    s = pd.to_numeric(df[col], errors="coerce").dropna()
    out = {f"{col}.mean": s.mean(), f"{col}.max": s.max(), f"{col}.last": s.iloc[-1]}
    if len(s) > 1:
        out[f"{col}.slope"] = np.polyfit(np.arange(len(s)), s.values, 1)[0]
    return out

def featurise_bundle(bundle: Path) -> dict:
    feats = {}
    hp = bundle / "host.parquet"
    if hp.exists():
        host = pd.read_parquet(hp)
        for col in ("cpu_pct","rss_bytes","num_fds","num_connections","num_threads",
                    "io_read_bytes","io_write_bytes"):
            feats.update(_agg(host, col))
    rp = bundle / "responses.parquet"
    if rp.exists():
        r = pd.read_parquet(rp)
        if len(r):
            req = pd.to_numeric(r.get("request_size_bytes"), errors="coerce")
            rsp = pd.to_numeric(r.get("response_size_bytes"), errors="coerce")
            dur = pd.to_numeric(r.get("duration_ns"), errors="coerce")
            feats["resp.count"] = float(len(r))
            feats["resp.resp_bytes.mean"] = rsp.mean(); feats["resp.resp_bytes.max"] = rsp.max()
            feats["resp.req_bytes.mean"] = req.mean()
            amp = rsp / req.replace(0, np.nan)
            feats["resp.amp.mean"] = amp.mean(); feats["resp.amp.max"] = amp.max()
            feats["resp.duration_ns.mean"] = dur.mean()
            feats["resp.distinct_endpoints"] = float(r.get("endpoint").nunique())
    return feats

## 3. Load the whole corpus into a feature table keyed by `(family, chain)`

In [ ]:
rows = []
for man in sorted(root.glob("crp_*/manifest.json")):
    m = json.loads(man.read_text())
    row = featurise_bundle(man.parent)
    row["family"] = m.get("family_id")
    row["chain"] = "solana" if m.get("chain") == "solana-agave" else m.get("chain")
    row["label"] = m.get("ground_truth_label") or m.get("label")
    rows.append(row)
df = pd.DataFrame(rows)
print(len(df), "bundles |", df["chain"].nunique(), "chains |", df["family"].nunique(), "families")
df["family"].value_counts()

## 4. Train + evaluate — two numbers, named separately

**In-distribution** (5-fold CV) is the easy number: families the model has seen on a chain.
**Cross-chain LOCO** is the honest one: a chain the model has *never trained on*.

In [ ]:
feat_cols = [c for c in df.columns if c not in ("family","chain","label")]
X = df[feat_cols].to_numpy(float); y = df["family"].to_numpy(); chains = df["chain"].to_numpy()

def fit(Xtr, ytr):
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                         min_samples_leaf=5, random_state=42)
    clf.fit(Xtr, ytr); return clf

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1s = [f1_score(y[te], fit(X[tr], y[tr]).predict(X[te]), average="macro") for tr, te in skf.split(X, y)]
print(f"[in-distribution] 5-fold CV family macro-F1: {np.mean(f1s):.3f}\n")
print("[cross-chain] leave-one-chain-out family macro-F1 (shared families only):")
for ch in sorted(set(chains)):
    tr, te = chains != ch, chains == ch
    shared = set(y[tr]) & set(y[te]); mask = te & np.isin(y, list(shared))
    if te.sum() < 3 or len(shared) < 2 or mask.sum() < 3:
        print(f"  {ch:<14} n/a (too few shared families)"); continue
    f1 = f1_score(y[mask], fit(X[tr], y[tr]).predict(X[mask]), average="macro")
    print(f"  {ch:<14} {f1:.3f}   (n={int(mask.sum())}, {len(shared)} shared families)")

## Earned autonomy — how to read the two numbers

The in-distribution number says the detector separates families it has **seen** on a chain. The cross-chain
number says whether that generalises to a chain it has **never** trained on.

Autonomy is *earned* only where the cross-chain number holds up:

- **High LOCO** on a chain → the mechanism signal genuinely transfers → deploy there with earned confidence.
- **Near-floor LOCO** → it does not → the honest move is to **gate** the detector to chains you have data
  for, and say so.

On this corpus the cross-chain number sits near the floor for protocol-distinct chains (Monero, Bitcoin,
Ethereum) and is trivially high only for wire-identical forks (Dogecoin/Litecoin inherit Bitcoin's exact
primitives). The corpus exists to let you **find that line** rather than assume it. That is the same
discipline behind NullRabbit's Validator Integrity Index: the method and its honest limits *are* the product.